# Deconvolution (calab: CaTune / CaDecon)

The explicit middle stage of the pipeline. Minian gives us denoised calcium
traces (`C`); here we **deconvolve** them into a per-unit estimate of neural
activity, which the CaMAP capstone then fuses with behavior. Because this step
is explicit, **Minian's own deconvolution was skipped** (see `tutorials/minian/`)
and CaMAP's built-in OASIS is bypassed -- calab owns deconvolution.

Deconvolution is provided by **calab**, which ships two browser apps:

- **CaTune** -- *interactive* tuning. You set the kernel and sparsity
  (`tau_rise`, `tau_decay`, `lambda`) by eye against your own traces, then export
  them. Use it to understand the knobs or when you want manual control.
- **CaDecon** -- *automated* deconvolution. It estimates per-cell parameters for
  you and returns the activity. The good default path.

Each tool can be used **two ways**, and both are laid out below:

1. **Explore (no data needed)** -- open the hosted app; it generates its own
   simulated traces *in the browser*. Pick the simulation yourself, play with the
   knobs. Nothing is sent from or saved back to this notebook.
2. **Your data** -- this notebook sends your Minian `C` traces to the app over a
   localhost bridge and saves the result to
   `data/sessions/<session>/deconv_out/activity.npy` -- exactly what the capstone
   reads.

> **Output:** `deconv_out/activity.npy`, shape `(n_cells, n_frames)`, with rows in
> the **same order as the Minian `C` traces**. The capstone's inject step relies
> on that order, so don't reorder the traces.

This notebook is the interactive companion to the headless scripts
`run_cadecon.py` / `run_catune.py` in this folder -- same calab calls, same
output. Use whichever you prefer.

## Setup

Pick the session, sampling rate, and which Minian trace to deconvolve, then load
the calab API. Keep `TRACE = "C"` (the raw denoised trace) -- `C_lp` is the
low-pass copy meant for display only.

In [ ]:
import webbrowser
from pathlib import Path

import numpy as np
from calab import (
    load_minian,            # Minian C.zarr -> (traces, metadata)
    tune,                   # open CaTune (interactive), return tuned params
    decon,                  # open CaDecon (automated), return activity
    run_deconvolution,      # FISTA deconvolution from explicit params
    bandpass_filter,        # optional pre-filter CaTune can enable
    deconvolve_from_export, # deconvolve from a downloaded CaTune export JSON
)

# ---- parameters -------------------------------------------------------------
SESSION = "prerecorded"   # which data/sessions/<name> to process
FS      = 20.0            # neural sampling rate (Hz)
TRACE   = "C"             # Minian trace to deconvolve (keep C; C_lp is display-only)
# -----------------------------------------------------------------------------

# Hosted calab apps (GitHub Pages); each generates its own simulated data.
CATUNE_URL  = "https://miniscope.github.io/CaLab/CaTune/"
CADECON_URL = "https://miniscope.github.io/CaLab/CaDecon/"


def _find_repo_root(start: Path) -> Path:
    """Walk up from *start* until we find the data/sessions tree."""
    for cand in [start.resolve(), *start.resolve().parents]:
        if (cand / "data" / "sessions").is_dir():
            return cand
    return start.resolve()


REPO_ROOT  = _find_repo_root(Path.cwd())
minian_out = REPO_ROOT / "data" / "sessions" / SESSION / "minian_out"
deconv_out = REPO_ROOT / "data" / "sessions" / SESSION / "deconv_out"
deconv_out.mkdir(parents=True, exist_ok=True)
czarr        = minian_out / f"{TRACE}.zarr"
activity_npy = deconv_out / "activity.npy"

print(f"session   : {SESSION}")
print(f"input     : {czarr}")
print(f"output    : {activity_npy}")

## Load your Minian traces

Reads `<trace>.zarr` from `minian_out/`. The row order is the Minian `C` unit
order, which the capstone relies on -- so we never reorder.

**Skip this cell** if you only want to *explore* with the apps' in-browser
simulated data (the "Explore" cells below need no input).

In [ ]:
if not czarr.exists():
    raise FileNotFoundError(
        f"{czarr} not found.\n"
        "Run the Minian notebook first, or `python scripts/get_data.py "
        f"--session {SESSION} --what minian_out`.\n"
        "(Or skip loading and use the 'Explore' cells with in-browser sim data.)"
    )

traces, meta = load_minian(str(czarr), trace_key=TRACE, fs=FS)
traces = np.asarray(traces)
print(f"Loaded {traces.shape[0]} cells x {traces.shape[1]} frames from {TRACE}.zarr")
print(f"metadata: {meta}")

## CaDecon -- automated deconvolution

CaDecon estimates a per-cell kernel and sparsity for you and runs the solver --
no manual tuning. This is the recommended default. The browser app does the work
and sends the activity back over the bridge.

### Explore in the browser (simulated data, no recording needed)

Opens the hosted CaDecon and lets you generate traces with its built-in
simulator and try the settings yourself. Nothing is loaded from or saved to this
notebook -- pure exploration.

In [ ]:
print(f"Opening CaDecon: {CADECON_URL}")
print("Use the app's built-in simulator to generate traces and try the "
      "deconvolution settings -- no data file needed.")
webbrowser.open(CADECON_URL)

### Run CaDecon on your traces

Sends the loaded Minian traces to CaDecon over a localhost bridge. With
`autorun=True` the solver starts on its own; otherwise click **Run** in the app.
The call **blocks** until the browser exports results back, then we save
`activity.npy`.

In [ ]:
result = decon(traces, fs=FS, autorun=True)   # opens CaDecon; waits for the export

if result is not None:
    activity = np.asarray(result.activity)
    np.save(activity_npy, activity)
    print(f"Saved deconvolved activity {activity.shape} -> {activity_npy}")
else:
    print("CaDecon returned nothing (timed out or cancelled) -- nothing saved.")

## CaTune -- interactive parameter tuning

CaTune lets you set the deconvolution kernel and sparsity by eye on your own
traces: rise time (`tau_rise`), decay time (`tau_decay`), sparsity (`lambda`),
and an optional bandpass pre-filter. Tune until the inferred activity looks
right, then **Export** -- the params come straight back to this notebook over the
bridge (no file download needed) and we run the deconvolution here.

### Explore in the browser (simulated data, no recording needed)

Opens the hosted CaTune so you can simulate traces in the browser and learn what
each knob does. Nothing is loaded from or saved to this notebook.

In [ ]:
print(f"Opening CaTune: {CATUNE_URL}")
print("Use the app's built-in simulator to generate traces and tune the "
      "parameters -- no data file needed.")
webbrowser.open(CATUNE_URL)

### Tune on your traces, then deconvolve

`tune()` opens CaTune with your loaded traces and **blocks until you click
Export** in the app; it returns the tuned params as a dict
(`tau_rise`, `tau_decay`, `lambda_`, `fs`, `filter_enabled`). We then run FISTA
deconvolution with those params (applying the bandpass pre-filter first if you
enabled it -- matching `deconvolve_from_export`) and save `activity.npy`.

In [ ]:
def deconvolve_with_params(traces, p):
    """Deconvolve using a CaTune params dict (mirrors deconvolve_from_export)."""
    tr = np.asarray(traces, dtype=np.float64)
    if p.get("filter_enabled"):
        tr = np.atleast_2d(tr).copy()
        for i in range(tr.shape[0]):
            tr[i] = bandpass_filter(tr[i], p["tau_rise"], p["tau_decay"], p["fs"])
        if tr.shape[0] == 1:
            tr = tr[0]
    return run_deconvolution(
        tr, fs=p["fs"], tau_r=p["tau_rise"], tau_d=p["tau_decay"], lam=p["lambda_"]
    )


params = tune(traces, fs=FS)   # opens CaTune; blocks until you EXPORT in the browser
print("Tuned params:", params)

if params is not None:
    activity = np.asarray(deconvolve_with_params(traces, params))
    np.save(activity_npy, activity)
    print(f"Saved deconvolved activity {activity.shape} -> {activity_npy}")
else:
    print("No params received (timed out or cancelled) -- nothing saved.")

### Alternative: apply a downloaded CaTune export

If you tuned in a separate browser session and **downloaded** the params JSON
(instead of using the live bridge above), point `PARAMS_JSON` at that file. This
mirrors `run_catune.py --params <export.json>`.

In [ ]:
PARAMS_JSON = ""   # path to a CaTune export .json (usually in your Downloads)

if PARAMS_JSON:
    activity = np.asarray(deconvolve_from_export(traces, PARAMS_JSON))
    np.save(activity_npy, activity)
    print(f"Saved deconvolved activity {activity.shape} -> {activity_npy}")
else:
    print("Set PARAMS_JSON to a downloaded CaTune export to use this path.")

## Verify and next step

Sanity-check the saved activity: it should be `(n_cells, n_frames)` and match the
Minian trace count and frame count. The neural timestamp file must have the same
frame count for the capstone to align behavior onto the neural clock.

In [ ]:
saved = np.load(activity_npy)
print(f"activity.npy: {saved.shape}  ({saved.dtype})")
if czarr.exists():
    n_cells, n_frames = traces.shape
    ok = saved.shape == (n_cells, n_frames)
    print(f"Minian {TRACE}: {n_cells} cells x {n_frames} frames -> "
          f"{'matches' if ok else 'MISMATCH'}")

print("\nNext: the CaMAP capstone reads this activity.npy. See capstone/.")